# cc13 behaviour change

`cc13` is always asked of new participants, and was asked to repeating participants in wave 4. Since our no-imputation analysis only includes participants from both waves 3 and 4, we then certainly have a repeating response from wave 4, and _sometimes_ have a new response from wave 3. If we consider `cc13_cumulative`, which takes the cumulative set union over consecutive waves for each individual, we definitely have a 'before' response for all individuals considered, though this response may have been prior to wave 3. 

Thus `cc13_cumulative` provides a measure of behaviour change across waves.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

from climate_attitudes import configure_mpl
from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

plt.rc("figure", dpi=150)

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, with_imputation=False)
resp = dataset.response.collect()

keep_pids = (
    dataset.participant.filter(pl.col("wave_3"), pl.col("wave_4"))
    .select("participant_id")
    .collect()
    .to_series()
    .implode()
)
resp = resp.filter(pl.col("participant_id").is_in(keep_pids))

In [ ]:
df = (
    resp.filter(pl.col("wave").is_in([3, 4]))
    .select("participant_id", "wave", "start_date", "cc13_cumulative")
    .sort(by=("participant_id", "wave"))
    .pivot("wave", index="participant_id", values="cc13_cumulative")
)

In [ ]:
df.filter(pl.col("3") != pl.col("4"))

In [ ]:
resp.filter(pl.col("cc13").is_null(), wave=3).select(
    "participant_id", "cc13", "cc13_cumulative"
)